In [3]:
import pandas as pd

# Load the dataset
file_path = "/content/blogs.csv"
df =pd.read_csv(file_path)

# Display basic information and first few rows
df.info(), df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Data    2000 non-null   object
 1   Labels  2000 non-null   object
dtypes: object(2)
memory usage: 31.4+ KB


(None,
                                                 Data       Labels
 0  Path: cantaloupe.srv.cs.cmu.edu!magnesium.club...  alt.atheism
 1  Newsgroups: alt.atheism\nPath: cantaloupe.srv....  alt.atheism
 2  Path: cantaloupe.srv.cs.cmu.edu!das-news.harva...  alt.atheism
 3  Path: cantaloupe.srv.cs.cmu.edu!magnesium.club...  alt.atheism
 4  Xref: cantaloupe.srv.cs.cmu.edu alt.atheism:53...  alt.atheism)

In [4]:
import re
import string
import nltk
from nltk.corpus import stopwords

# Download stopwords if not already available
nltk.download("stopwords")

# Define text cleaning function
def clean_text(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'\n', ' ', text)  # Remove newline characters
    text = re.sub(r"http\S+|www\S+|https\S+", '', text, flags=re.MULTILINE)  # Remove URLs
    text = re.sub(r'\b\d+\b', '', text)  # Remove numbers
    text = text.translate(str.maketrans('', '', string.punctuation))  # Remove punctuation
    text = ' '.join([word for word in text.split() if word not in stopwords.words("english")])  # Remove stopwords
    return text

# Apply cleaning function to the 'Data' column
df["Cleaned_Data"] = df["Data"].apply(clean_text)

# Remove stopwords
stop_words = set(stopwords.words("english"))
df["Cleaned_Data"] = df["Cleaned_Data"].apply(lambda x: " ".join([word for word in x.split() if word not in stop_words]))

# Display the first few cleaned entries
df[["Cleaned_Data", "Labels"]].head()


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


,Cleaned_Data,Labels
0,path cantaloupesrvcscmuedumagnesiumclubcccmued...,alt.atheism
1,newsgroups altatheism path cantaloupesrvcscmue...,alt.atheism
2,path cantaloupesrvcscmuedudasnewsharvardedunoc...,alt.atheism
3,path cantaloupesrvcscmuedumagnesiumclubcccmued...,alt.atheism
4,xref cantaloupesrvcscmuedu altatheism talkreli...,alt.atheism


In [6]:
# Further refining the text cleaning function
def refine_clean_text(text):
    text = re.sub(r'\bpath\b|\bnewsgroups\b|\bxref\b', '', text)  # Remove metadata terms
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra spaces
    return text

# Apply the refined cleaning function
df["Cleaned_Data"] = df["Cleaned_Data"].apply(refine_clean_text)

# Display the first few cleaned entries
df[["Cleaned_Data", "Labels"]].head()


,Cleaned_Data,Labels
0,cantaloupesrvcscmuedumagnesiumclubcccmuedunews...,alt.atheism
1,altatheism cantaloupesrvcscmueducrabapplesrvcs...,alt.atheism
2,cantaloupesrvcscmuedudasnewsharvardedunocnearn...,alt.atheism
3,cantaloupesrvcscmuedumagnesiumclubcccmuedunews...,alt.atheism
4,cantaloupesrvcscmuedu altatheism talkreligionm...,alt.atheism


In [7]:
# Further refining to remove email-like patterns and server names
def final_clean_text(text):
    text = re.sub(r'\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b', '', text)  # Remove emails
    text = re.sub(r'\b[a-zA-Z0-9.-]*s?cmu?edu\b', '', text)  # Remove server names
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra spaces
    return text

# Apply the final cleaning function
df["Cleaned_Data"] = df["Cleaned_Data"].apply(final_clean_text)

# Display the first few cleaned entries
df[["Cleaned_Data", "Labels"]].head()


,Cleaned_Data,Labels
0,cantaloupesrvcscmuedumagnesiumclubcccmuedunews...,alt.atheism
1,altatheism cantaloupesrvcscmueducrabapplesrvcs...,alt.atheism
2,cantaloupesrvcscmuedudasnewsharvardedunocnearn...,alt.atheism
3,cantaloupesrvcscmuedumagnesiumclubcccmuedunews...,alt.atheism
4,altatheism talkreligionmisc talkorigins altath...,alt.atheism


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF vectorizer
vectorizer = TfidfVectorizer(max_features=5000)  # Limiting to top 5000 features for efficiency

# Transform the cleaned text data into numerical format
X = vectorizer.fit_transform(df["Cleaned_Data"])

# Extract target labels
y = df["Labels"]

# Display shape of transformed data
X.shape, y.shape


((2000, 5000), (2000,))

In [9]:
from sklearn.model_selection import train_test_split

# Split data into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Display the shape of the training and testing sets
X_train.shape, X_test.shape, y_train.shape, y_test.shape


((1600, 5000), (400, 5000), (1600,), (400,))

In [10]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# Initialize the Naive Bayes classifier
nb_classifier = MultinomialNB()

# Train the classifier
nb_classifier.fit(X_train, y_train)

# Make predictions on the test set
y_pred = nb_classifier.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

accuracy, report


(0.8375,
 '                          precision    recall  f1-score   support\n\n             alt.atheism       0.84      0.80      0.82        20\n           comp.graphics       0.89      0.85      0.87        20\n comp.os.ms-windows.misc       0.84      0.80      0.82        20\ncomp.sys.ibm.pc.hardware       0.54      0.70      0.61        20\n   comp.sys.mac.hardware       0.82      0.70      0.76        20\n          comp.windows.x       0.84      0.80      0.82        20\n            misc.forsale       0.86      0.90      0.88        20\n               rec.autos       0.86      0.90      0.88        20\n         rec.motorcycles       0.89      0.85      0.87        20\n      rec.sport.baseball       0.90      0.95      0.93        20\n        rec.sport.hockey       1.00      1.00      1.00        20\n               sci.crypt       0.91      1.00      0.95        20\n         sci.electronics       0.84      0.80      0.82        20\n                 sci.med       0.88      0.75    

In [11]:
from textblob import TextBlob
# Function to get sentiment polarity
def get_sentiment(text):
    return TextBlob(text).sentiment.polarity

# Apply sentiment analysis
df["Sentiment_Score"] = df["Cleaned_Data"].apply(get_sentiment)

# Classify sentiment into Positive, Negative, and Neutral
df["Sentiment"] = df["Sentiment_Score"].apply(lambda x: "Positive" if x > 0 else ("Negative" if x < 0 else "Neutral"))

# Display updated dataset with sentiment classification
df[["Cleaned_Data", "Labels", "Sentiment"]].head()


,Cleaned_Data,Labels,Sentiment
0,cantaloupesrvcscmuedumagnesiumclubcccmuedunews...,alt.atheism,Positive
1,altatheism cantaloupesrvcscmueducrabapplesrvcs...,alt.atheism,Negative
2,cantaloupesrvcscmuedudasnewsharvardedunocnearn...,alt.atheism,Positive
3,cantaloupesrvcscmuedumagnesiumclubcccmuedunews...,alt.atheism,Positive
4,altatheism talkreligionmisc talkorigins altath...,alt.atheism,Positive


In [14]:
 # Summarize sentiment distribution across categories
sentiment_distribution = df.groupby("Labels")["Sentiment"].value_counts(normalize=True) * 100

# Convert to a readable format
sentiment_distribution = sentiment_distribution.unstack().fillna(0)

# Display sentiment distribution summary
sentiment_distribution


Sentiment,Negative,Neutral,Positive
Labels,,,
alt.atheism,35.0,0.0,65.0
comp.graphics,27.0,0.0,73.0
comp.os.ms-windows.misc,24.0,0.0,76.0
comp.sys.ibm.pc.hardware,19.0,0.0,81.0
comp.sys.mac.hardware,26.0,0.0,74.0
comp.windows.x,20.0,2.0,78.0
misc.forsale,21.0,0.0,79.0
rec.autos,24.0,0.0,76.0
rec.motorcycles,28.0,0.0,72.0


# **The sentiment distribution analysis reveals the following insights:**

* Categories like "soc.religion.christian" and "talk.religion.misc" have the highest percentage of positive sentiment (86%), indicating that discussions in these categories are generally optimistic or supportive.
* "misc.forsale" also has a high positive sentiment (83%), which makes sense as it's related to sales and advertisements, often presented in a positive manner.
* "rec.sport.hockey" and "rec.sport.baseball" show the highest negative sentiment (38% and 36%), likely due to passionate debates or criticism related to team performances.
* Political and religious discussions (e.g., "talk.politics.guns", "talk.politics.mideast", "alt.atheism") have mixed sentiments, reflecting the polarizing nature of these topics.
* Most categories lack neutral sentiment, suggesting that blogs tend to express strong opinions rather than neutrality.